# Reinsurance Loss Development - EDA & Prototyping

Exploratory Data Analysis notebook for loss triangle ingestion, visualization, and feature engineering prototyping.

## Section 7: Next Steps

Once the core modules are implemented, the following enhancements can be added:

1. **Load real data**: Replace manual sample with actual parse_csv() output
2. **Validation exploration**: Identify data quality issues using validate()
3. **Feature distribution**: Analyze feature panel statistics, correlations, outliers
4. **Model diagnostics**: Residuals, feature importance, prediction intervals
5. **Stress testing**: Apply models to out-of-sample triangles
6. **Documentation**: Generate reserves for management reporting

### TODOs

- [ ] Implement `ingestion/triangles.py` functions
- [ ] Implement `features/engineering.py` feature panel
- [ ] Implement `models/chainladder.py` method
- [ ] Implement `models/mlmodel.py` with chosen architecture
- [ ] Load real data and run full analysis
- [ ] Create publication-quality visualizations

In [ ]:
# TODO: Once ML model is implemented, compare predictions

# For now, create synthetic comparison
print("Hypothetical ML vs Chain-Ladder Comparison:")
print("=" * 70)

cl_estimates = {
    2020: 1700000,  # CL projection
    2021: 1500000,
    2022: 1420000,
    2023: 1050000,  # Most uncertain
}

ml_estimates = {
    2020: 1680000,  # ML slightly lower on recent
    2021: 1520000,
    2022: 1410000,
    2023: 1200000,  # ML projects higher development
}

results = []
for year in [2020, 2021, 2022, 2023]:
    cl = cl_estimates[year]
    ml = ml_estimates[year]
    divergence_pct = abs(ml - cl) / cl * 100
    flagged = divergence_pct > 15  # Threshold
    
    results.append({
        'year': year,
        'chain_ladder': cl,
        'ml_model': ml,
        'divergence_pct': divergence_pct,
        'flagged': flagged
    })

results_df = pd.DataFrame(results)

print(f"{'Year':<6} {'Chain-Ladder':>15} {'ML Model':>15} {'Divergence %':>15} {'Flagged':>10}")
print("-" * 70)
for _, row in results_df.iterrows():
    flag = "⚠️  YES" if row['flagged'] else "  No"
    print(f"{int(row['year']):<6} ${row['chain_ladder']/1e6:>14.2f}M ${row['ml_model']/1e6:>14.2f}M {row['divergence_pct']:>14.1f}% {flag:>10}")

print("\n" + "=" * 70)
print("Interpretation:")
print("- Rows with divergence > 15% are flagged for actuarial review")
print("- Actuaries investigate why ML and CL differ significantly")
print("- Decision: trust ML, trust CL, or blend estimates")

## Section 6: Model Comparison Prototype

Compare chain-ladder projections to a hypothetical ML model and visualize divergences.

In [ ]:
# TODO: Once ChainLadder is implemented, use it to project
# chainladder_model = ChainLadder()
# chainladder_model.fit(triangle)
# ultimate_losses = chainladder_model.predict(triangle)
# ibnr_reserves = chainladder_model.get_ibnr(triangle, ultimate_losses)

# Manual prototype chain-ladder projection
print("Manual Chain-Ladder Projection Prototype:")
print("=" * 60)

# Step 1: Calculate development factors (using 2019, fully mature row)
mature_row = sample_cumulative[0, :]  # 2019 is the most complete
factors = []
for j in range(1, len(mature_row)):
    if not np.isnan(mature_row[j]) and not np.isnan(mature_row[j-1]) and mature_row[j-1] > 0:
        factor = mature_row[j] / mature_row[j-1]
        factors.append(factor)
        print(f"Factor {dev_periods[j-1]:2d}→{dev_periods[j]:2d} months: {factor:.4f}")

print("\n" + "=" * 60)
print("Projections for recent cohorts:")
print("=" * 60)

# Step 2: Project recent cohorts
for i, year in enumerate(origin_years[1:], 1):  # Skip 2019 (fully developed)
    current_row = sample_cumulative[i, :]
    
    # Find last non-NaN value
    last_valid_idx = None
    for j in range(len(current_row) - 1, -1, -1):
        if not np.isnan(current_row[j]):
            last_valid_idx = j
            break
    
    if last_valid_idx is not None and last_valid_idx < len(dev_periods) - 1:
        current_cum = current_row[last_valid_idx]
        
        # Project to ultimate by applying remaining factors
        projected_cum = current_cum
        for j in range(last_valid_idx + 1, len(dev_periods)):
            factor_idx = j - 1
            if factor_idx < len(factors):
                projected_cum *= factors[factor_idx]
        
        ibnr = projected_cum - current_cum
        
        print(f"\n{year} (Last observed: ${current_cum/1e6:.2f}M at {dev_periods[last_valid_idx]} months)")
        print(f"  → Projected Ultimate: ${projected_cum/1e6:.2f}M")
        print(f"  → IBNR Reserve: ${ibnr/1e6:.2f}M")

## Section 5: Chain-Ladder Projection Prototype

Prototype chain-ladder method to project ultimate losses for recent cohorts.

In [ ]:
# TODO: Once build_feature_panel is implemented, generate features
# feature_panel = build_feature_panel(triangle)
# print(f"Feature Panel Shape: {feature_panel.shape}")
# print(f"Features:\n{feature_panel.head(10)}")

# Manual preview of what feature panel should look like
print("Manual Feature Panel Preview:")
print("=" * 80)

rows = []
for i, year in enumerate(origin_years):
    for j, dev_period in enumerate(dev_periods):
        if not np.isnan(sample_cumulative[i, j]):
            cum_loss = sample_cumulative[i, j]
            
            # Compute incremental loss
            if j == 0:
                incr_loss = cum_loss
            else:
                prior_cum = sample_cumulative[i, j-1]
                if not np.isnan(prior_cum):
                    incr_loss = cum_loss - prior_cum
                else:
                    incr_loss = np.nan
            
            # Development age
            dev_age = dev_period / 12
            
            rows.append({
                'origin_year': year,
                'dev_period': dev_period,
                'dev_age': dev_age,
                'cumulative_loss': cum_loss,
                'incremental_loss': incr_loss,
                'log_cum_loss': np.log(cum_loss + 1),
            })

feature_panel_manual = pd.DataFrame(rows)
print(feature_panel_manual.head(15))
print(f"\nTotal rows (cells): {len(feature_panel_manual)}")

## Section 4: Feature Engineering Prototype

Prototype feature engineering: convert triangle to row-per-cell panel with engineered features.

In [ ]:
# TODO: Implement development factor calculation
# Factor[dev_period] = Sum(cumulative_loss[:, dev_period]) / Sum(cumulative_loss[:, dev_period-1])

# Manual calculation on sample data
print("Manual Development Factors Calculation:")
print("=" * 50)

for j in range(1, len(dev_periods)):
    # Get non-NaN values from column j and j-1
    col_curr = sample_cumulative[:, j]
    col_prev = sample_cumulative[:, j-1]
    
    # Use only rows where both are non-NaN
    valid_mask = ~(np.isnan(col_curr) | np.isnan(col_prev))
    
    if valid_mask.sum() > 0:
        sum_curr = np.nansum(col_curr)
        sum_prev = np.nansum(col_prev)
        
        if sum_prev > 0:
            factor = sum_curr / sum_prev
            print(f"Factor {dev_periods[j-1]:2d} → {dev_periods[j]:2d} months: {factor:.4f}")

print("\nInterpretation: Factors > 1.0 = claims still developing")
print("Factors → 1.0 = claims mature, little new development expected")

## Section 3: Compute Development Factors

Calculate age-to-age development factors (the foundation of chain-ladder method).

In [ ]:
# TODO: Plot development curves for each origin year
# Each line represents one accident year; x-axis is development age, y-axis is cumulative loss

# Manual preview of what the plot should show
fig, ax = plt.subplots(figsize=(10, 6))

for i, year in enumerate(origin_years):
    # Get non-NaN values for this row
    losses = sample_cumulative[i, :]
    valid_idx = ~np.isnan(losses)
    dev_valid = np.array(dev_periods)[valid_idx]
    losses_valid = losses[valid_idx]
    
    ax.plot(dev_valid, losses_valid, marker='o', label=f"Year {year}")

ax.set_xlabel("Development Period (months)", fontsize=12)
ax.set_ylabel("Cumulative Loss ($)", fontsize=12)
ax.set_title("Loss Development Patterns by Accident Year", fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("TODO: Enhance visualization with interactive Plotly once data is loaded")

## Section 2: Visualize Development Patterns

Visualize how losses develop across cohorts. Early development periods should show steep growth, later periods should flatten.

In [ ]:
# TODO: Once parse_csv is implemented, load the sample triangle
# triangle = parse_csv(Path("../data/raw/sample_triangle.csv"))
# print(f"Line of Business: {triangle.line_of_business}")
# print(f"Origin Years: {triangle.origin_years}")
# print(f"Development Periods: {triangle.development_periods}")
# print(f"Cumulative Losses:\n{triangle.cumulative_losses}")

# For now, manually create sample data to preview
print("TODO: Load sample_triangle.csv using parse_csv() once implemented")

# Manual preview of expected data structure
sample_cumulative = np.array([
    [1000000, 1580000, 1720000, 1785000, 1820000],
    [950000, 1501000, 1640000, 1708000, np.nan],
    [920000, 1460000, 1600000, np.nan, np.nan],
    [880000, 1408000, np.nan, np.nan, np.nan],
    [850000, np.nan, np.nan, np.nan, np.nan],
])

origin_years = [2019, 2020, 2021, 2022, 2023]
dev_periods = [12, 24, 36, 48, 60]

print(f"Sample Triangle (5x5):")
print(f"Origin Years: {origin_years}")
print(f"Development Periods: {dev_periods} months")
print(f"Cumulative Losses:\n{sample_cumulative}")

## Section 1: Load Sample Triangle

Load the synthetic loss triangle from data/raw/sample_triangle.csv and explore its structure.

In [ ]:
import sys
from pathlib import Path

# Add project root to path for imports
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# TODO: Import once implemented
# from ingestion.triangles import parse_csv, parse_excel, validate, to_dataframe, LossTriangle
# from features.engineering import build_feature_panel
# from models.chainladder import ChainLadder

print("Setup complete. Modules ready for import.")